# Candidate Generation Optimization: Marginal Precision Strategy
Alessio Pizzini, David Ravelli
## Stage 1 Retrieval Analysis for XGBoost Reranking

In a Two-Stage Recommender System, the Candidate Generation (Retrieval) stage is critical. The goal is to maximize the **Recall Ceiling** (ensuring the relevant items are in the candidate pool) while minimizing the **Candidate Set Size** (to prevent memory bottleneck and latency issues in the XGBoost Reranker).

### The Challenge
A naive approach uses a fixed cutoff $K$ for all base models. However, different algorithms have different precision decay rates. For instance, an accurate model like `SLIM ElasticNet` might still provide highly relevant items at rank 100, while a high-coverage model like `LightFM` might mostly introduce noise after rank 30.

### Solution: Marginal Precision Thresholding
Instead of a global cutoff, we implemented a **Marginal Precision Strategy**:
1. We analyze the marginal precision of each model at every rank $k$.
2. We set a **Target Precision Threshold** (e.g., 2.0% or 4.0%).
3. We assign a personalized cutoff to each model exactly where its marginal precision drops below the target threshold.

This approach guarantees: extracting the maximum valuable signal from each model while strictly controlling the noise-to-signal ratio fed into the XGBoost reranker.

In [ ]:
!pip install protobuf==3.20.3
!pip install lightfm
!pip install implicit

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

INPUT_DIR = "/kaggle/input/recommender-systems-2025-challenge-polimi"

print("File disponibili in input:")
for dirname, _, filenames in os.walk(INPUT_DIR):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv(f"{INPUT_DIR}/data_train.csv")

print(df.shape)
print(df.info())
df.head()

In [ ]:
!git clone https://github.com/remaplab/RecSys_Course_AT_PoliMi

In [ ]:
import sys
sys.path.append('./RecSys_Course_AT_PoliMi')
print(sys.path)

In [ ]:
from scipy.sparse import coo_matrix, csr_matrix

n_users = df['row'].max() + 1
n_items = df['col'].max() + 1


print("Numero di utenti unici:", len(df['row'].unique()))

A_coo = coo_matrix((np.ones(len(df), dtype=np.int8), (df['row'], df['col'])),
                   shape=(n_users, n_items))

URM_all = A_coo.tocsr()

In [ ]:
from Evaluation.Evaluator import EvaluatorHoldout
from Data_manager.split_functions.split_train_validation_random_holdout import split_train_in_two_percentage_global_sample

print(f"URM_all total interactions: {URM_all.nnz}")

URM_fit_models, URM_test = split_train_in_two_percentage_global_sample(URM_all, train_percentage=0.8)


print(f"URM_all: {URM_all.nnz} interazioni")
print(f"URM_fit_models (Base Models Train): {URM_fit_models.nnz} interazioni")
print(f"URM_train_xgb (XGBoost Test): {URM_test.nnz} interazioni")

# Base Models Training

In [ ]:
from Recommenders.FactorizationMachines.LightFMRecommender import LightFMCFRecommender

lightFM_params = {
                 'loss': 'warp',
                 'n_components': 157,
                 'item_alpha': 0.001019490618869453,
                 'user_alpha': 2.2744836269942957e-05,
                 'learning_rate': 0.012460981535727717,
                 'sgd_mode': 'adadelta',
                 'epochs': 74
              }

lightFM_recommender = LightFMCFRecommender(URM_fit_models)
lightFM_recommender.fit(**lightFM_params)

In [ ]:
import time
import torch
import torch.nn.functional as F
import numpy as np
import math
from tqdm import tqdm

from Recommenders.Neural.MultVAE_PyTorch_Recommender import MultVAERecommender_PyTorch

try:
    from Recommenders.Neural.architecture_utils import generate_autoencoder_architecture
except ImportError:
    def generate_autoencoder_architecture(encoding_size, n_items, next_layer_size_multiplier, max_parameters, max_n_hidden_layers):
        enc_dims = [n_items]
        for _ in range(max_n_hidden_layers):
            next_dim = int(enc_dims[-1] / next_layer_size_multiplier)
            if next_dim < encoding_size: break
            enc_dims.append(next_dim)
        enc_dims.append(int(encoding_size))
        return enc_dims

class MultVAERecommender_PyTorch_Fixed(MultVAERecommender_PyTorch):

    def fit(self, epochs=10, batch_size=500, total_anneal_steps=200000,
            learning_rate=1e-3, l2_reg=0.01, dropout=0.5, anneal_cap=0.2,
            sgd_mode="adam", encoding_size=50, next_layer_size_multiplier=2,
            max_parameters=np.inf, max_n_hidden_layers=3, **earlystopping_kwargs):

        encoding_size = int(encoding_size)
        batch_size = int(batch_size)
        total_anneal_steps = int(total_anneal_steps)

        p_dims = generate_autoencoder_architecture(encoding_size, self.n_items, next_layer_size_multiplier, max_parameters, max_n_hidden_layers)
        p_dims = [int(d) for d in p_dims]

        self._print(f"Architecture: {p_dims}")

        super().fit(epochs=epochs, batch_size=batch_size, dropout=dropout,
                    learning_rate=learning_rate, total_anneal_steps=total_anneal_steps,
                    anneal_cap=anneal_cap, p_dims=p_dims, l2_reg=l2_reg,
                    sgd_mode=sgd_mode, **earlystopping_kwargs)

    def _run_epoch(self, num_epoch):
        num_batches_per_epoch = math.ceil(len(self.warm_user_ids) / self.batch_size)
        if self.verbose:
            batch_iterator = tqdm(range(0, num_batches_per_epoch))
        else:
            batch_iterator = range(0, num_batches_per_epoch)

        self._model.train()
        epoch_loss = 0

        for _ in batch_iterator:
            self._optimizer.zero_grad()

            u_indices_np = np.random.choice(self.warm_user_ids, size=self.batch_size)
            user_batch_matrix = self.URM_train[u_indices_np]

            user_batch_tensor = torch.sparse_csr_tensor(
                user_batch_matrix.indptr, user_batch_matrix.indices, user_batch_matrix.data,
                size=user_batch_matrix.shape, dtype=torch.float32, device=self.device, requires_grad=False
            ).to_dense()

            logits, KL, mu_q, std_q, epsilon, sampled_z = self._model.forward(user_batch_tensor)
            log_softmax_var = F.log_softmax(logits, dim=1)
            neg_ll = - torch.mean(torch.sum(log_softmax_var * user_batch_tensor, dim=1))
            l2_reg_loss = self._model.get_l2_reg()

            if self.total_anneal_steps > 0:
                anneal = min(self.anneal_cap, 1. * self.update_count / self.total_anneal_steps)
            else:
                anneal = self.anneal_cap

            loss = neg_ll + anneal * KL + l2_reg_loss * self.l2_reg
            self.update_count += 1
            loss.backward()
            epoch_loss += loss.item()
            self._optimizer.step()

        if self.verbose:
            self._print("Loss {:.2E}".format(epoch_loss))
        self._model.eval()

    def _compute_item_score(self, user_id_array, items_to_compute=None):

        user_batch_matrix = self.URM_train[user_id_array]

        user_batch_tensor = torch.sparse_csr_tensor(
            user_batch_matrix.indptr, user_batch_matrix.indices, user_batch_matrix.data,
            size=user_batch_matrix.shape, dtype=torch.float32, device=self.device, requires_grad=False
        ).to_dense()


        with torch.no_grad():
            self._model.eval()
            logits, _, _, _, _, _ = self._model.forward(user_batch_tensor)

        item_scores_to_compute = logits.cpu().detach().numpy()

        if items_to_compute is not None:
            item_scores = - np.ones((len(user_id_array), self.n_items)) * np.inf
            item_scores[:, items_to_compute] = item_scores_to_compute[:, items_to_compute]
        else:
            item_scores = item_scores_to_compute

        return item_scores

In [ ]:
# 1. Definisco i parametri ottimali (copiati dai tuoi risultati)
vae_params = {
    'learning_rate': 0.0003696840239724747,
     'l2_reg': 0.0006675069550995708,
     'batch_size': 256,
     'dropout': 0.36764426901664576,
     'total_anneal_steps': 18522,
     'anneal_cap': 0.14211955419238365,
     'encoding_size': 545,
     'next_layer_size_multiplier': 10,
     'max_n_hidden_layers': 1
    }


multVAE_recommender = MultVAERecommender_PyTorch_Fixed(URM_fit_models, use_gpu=True)

multVAE_recommender.fit(
    epochs=65,
    learning_rate=vae_params['learning_rate'],
    l2_reg=vae_params['l2_reg'],
    batch_size=int(vae_params['batch_size']),
    dropout=vae_params['dropout'],
    anneal_cap=vae_params['anneal_cap'],
    total_anneal_steps=int(vae_params['total_anneal_steps']),
    encoding_size=int(vae_params['encoding_size']),
    next_layer_size_multiplier=int(vae_params['next_layer_size_multiplier']),
    max_n_hidden_layers=int(vae_params['max_n_hidden_layers']),
    sgd_mode='adam'
)

print("Training MultVAE completato con i Best Params.")

In [ ]:
# RP3beta - GRAPH SHARPNER
from Recommenders.Recommender_utils import check_matrix, similarityMatrixTopK
from Recommenders.Similarity.Compute_Similarity_Python import Incremental_Similarity_Builder
from Utils.seconds_to_biggest_unit import seconds_to_biggest_unit
from Recommenders.GraphBased.RP3betaRecommender import RP3betaRecommender

rp3bSharp_params = {"alpha": 1.3, "beta": 0.6, "topK": 200, "normalize_similarity": True}

RP3BetaSharp_recommender = RP3betaRecommender(URM_fit_models)
RP3BetaSharp_recommender.fit(
    **rp3bSharp_params,
    min_rating = 0,
    implicit = False,
)

In [ ]:
# RP3beta - HIGH PRECISION
from Recommenders.Recommender_utils import check_matrix, similarityMatrixTopK
from Recommenders.Similarity.Compute_Similarity_Python import Incremental_Similarity_Builder
from Utils.seconds_to_biggest_unit import seconds_to_biggest_unit
from Recommenders.GraphBased.RP3betaRecommender import RP3betaRecommender

rp3bHighPrec_params = {"alpha": 0.3, "beta": 0.15, "topK": 40, "normalize_similarity": True}

RP3BetaHighPrec_recommender = RP3betaRecommender(URM_fit_models)
RP3BetaHighPrec_recommender.fit(
    **rp3bHighPrec_params,
    min_rating = 0,
    implicit = False,
)

In [ ]:
import numpy as np
import scipy.sparse as sps

try:
    from Recommenders.BaseRecommender import BaseRecommender
except ImportError:
    from BaseRecommender import BaseRecommender



class ImplicitALSRecommender(BaseRecommender):
    """
    Wrapper for Implicit ALS compatible with Maurizio Ferrari Dacrema's Framework.
    """
    RECOMMENDER_NAME = "ImplicitALSRecommender"

    def __init__(self, URM_train, verbose=True):
        super(ImplicitALSRecommender, self).__init__(URM_train, verbose=verbose)

    def fit(self,
            factors=64,
            regularization=0.01,
            iterations=20,
            use_gpu=False,
            dtype=np.float32,
            calculate_training_loss=False,
            random_state=42,
            alpha_val=1.0):

        try:
            from implicit.als import AlternatingLeastSquares
        except ImportError as e:
            raise ImportError("Install the 'implicit' package (pip install implicit). Error: {}".format(e))


        self.factors = factors
        self.regularization = regularization
        self.iterations = iterations
        self.use_gpu = use_gpu
        self.dtype = dtype
        self.calculate_training_loss = calculate_training_loss
        self.random_state = random_state
        self.alpha_val = alpha_val


        URM_train_csr = self.URM_train.copy().astype(self.dtype)
        item_user_data = URM_train_csr.T.tocsr()

        if self.alpha_val is not None and self.alpha_val != 1.0:
            item_user_data.data *= self.alpha_val

        self.model = AlternatingLeastSquares(
            factors=self.factors,
            regularization=self.regularization,
            iterations=self.iterations,
            calculate_training_loss=self.calculate_training_loss,
            use_gpu=self.use_gpu,
            random_state=self.random_state
        )

        if self.verbose:
            print(f"{self.RECOMMENDER_NAME}: Training with factors={self.factors}, alpha={self.alpha_val}...")

        self.model.fit(item_user_data)


        factor_A = self.model.user_factors
        factor_B = self.model.item_factors

        if self.use_gpu:
            try:
                import cupy as cp
                if isinstance(factor_A, cp.ndarray): factor_A = cp.asnumpy(factor_A)
                if isinstance(factor_B, cp.ndarray): factor_B = cp.asnumpy(factor_B)
            except ImportError:
                pass

        factor_A = factor_A.astype(np.float32)
        factor_B = factor_B.astype(np.float32)


        if factor_A.shape[0] == self.n_users:
            self.user_factors = factor_A
            self.item_factors = factor_B
        elif factor_A.shape[0] == self.n_items:
            self.user_factors = factor_B
            self.item_factors = factor_A
        else:
            self.user_factors = factor_A
            self.item_factors = factor_B

        if self.verbose:
            print(f"{self.RECOMMENDER_NAME}: Training complete.")


    def _compute_item_score(self, user_id_array, items_to_compute=None):
        """
        Metodo richiesto dal Framework per calcolare le predizioni.
        """
        batch_user_factors = self.user_factors[user_id_array]

        if items_to_compute is not None:
            batch_item_factors = self.item_factors[items_to_compute]
        else:
            batch_item_factors = self.item_factors

        scores = np.dot(batch_user_factors, batch_item_factors.T)

        return scores


    def save_model(self, folder_path, file_name=None):
        """
        Salvataggio compatibile con il framework DataIO
        """
        if file_name is None:
            file_name = self.RECOMMENDER_NAME

        self._print("Saving model to file '{}'".format(folder_path + file_name))

        data_dict = {
            "user_factors": self.user_factors,
            "item_factors": self.item_factors,
            "factors": self.factors,
            "regularization": self.regularization,
            "alpha_val": self.alpha_val
        }

        try:
            from Recommenders.DataIO import DataIO
            dataIO = DataIO(folder_path=folder_path)
            dataIO.save_data(file_name=file_name, data_dict_to_save=data_dict)
        except ImportError:
            np.savez(folder_path + file_name, **data_dict)

        self._print("Saving complete")

    def load_model(self, folder_path, file_name=None):
        if file_name is None:
            file_name = self.RECOMMENDER_NAME

        self._print("Loading model from file '{}'".format(folder_path + file_name))

        data_dict = None

        try:
            from Recommenders.DataIO import DataIO
            dataIO = DataIO(folder_path=folder_path)
            data_dict = dataIO.load_data(file_name=file_name)
        except (ImportError, FileNotFoundError):
            # Fallback numpy load
            print("DataIO not found or file not found via DataIO, trying numpy.load")
            try:
                data_dict = np.load(folder_path + file_name + ".npz")
            except:
                data_dict = np.load(folder_path + file_name)

        self.user_factors = data_dict["user_factors"]
        self.item_factors = data_dict["item_factors"]
        self.factors = int(data_dict["factors"])

        if "regularization" in data_dict:
            self.regularization = data_dict["regularization"]
        if "alpha_val" in data_dict:
            self.alpha_val = data_dict["alpha_val"]

        self._print("Loading complete")

In [ ]:
iALS_recommender = ImplicitALSRecommender(URM_fit_models, verbose=True)
iALS_recommender.fit(
    factors=66,
    regularization=0.01758008114988,
    iterations=19,
    use_gpu=False,
    alpha_val=13.273206339452924,
    dtype=np.float32
)

In [ ]:
#SLIM EN
from Recommenders.SLIM.SLIMElasticNetRecommender import SLIMElasticNetRecommender

SLIM_EN_recommender = SLIMElasticNetRecommender(URM_fit_models)
SLIM_EN_recommender.fit(
    l1_ratio = 0.21208828301360144,
    alpha = 0.0010731928833306275,
    positive_only = True,
    topK = 494
)

In [ ]:
#SLIM EN - NEGATIVES
from Recommenders.SLIM.SLIMElasticNetRecommender import SLIMElasticNetRecommender

slimENneg_params = {
            "topK": 200,
            "l1_ratio": 0.1,
            "alpha": 0.001,
            "positive_only": False
        }

SLIM_EN_neg_recommender = SLIMElasticNetRecommender(URM_fit_models)
SLIM_EN_neg_recommender.fit(
    **slimENneg_params
)



In [ ]:
from Recommenders.EASE_R.EASE_R_Recommender import EASE_R_Recommender

EASE_R_recommender = EASE_R_Recommender(URM_fit_models)
EASE_R_recommender.fit(topK=725, l2_norm=375.6601396119732, normalize_matrix=False)


In [ ]:
import numpy as np
import scipy.sparse as sps

try:
    from Recommenders.BaseRecommender import BaseRecommender
except ImportError:
    from BaseRecommender import BaseRecommender

try:
    from implicit.nearest_neighbours import TFIDFRecommender
except ImportError as e:
    raise ImportError("Install the 'implicit' package (pip install implicit). Error: {}".format(e))


class ImplicitTFIDFRecommender(BaseRecommender):
    """
    Wrapper for Implicit TFIDFRecommender compatible with Dacrema's Framework.
    """
    RECOMMENDER_NAME = "ImplicitTFIDFRecommender"

    def __init__(self, URM_train, verbose=True):
        super(ImplicitTFIDFRecommender, self).__init__(URM_train, verbose=verbose)

    def fit(self, K=20, num_threads=0):
        self.K = K
        self.num_threads = num_threads

        self.model = TFIDFRecommender(K=self.K, num_threads=self.num_threads)


        if self.URM_train.shape[0] == self.n_users:
            train_data = self.URM_train.tocsr()
        else:
            train_data = self.URM_train.T.tocsr()

        self.model.fit(train_data)

        # 4. Estrazione della Matrice di Similarità
        self.W_sparse = self.model.similarity

        # Sanity Check finale: La similarità deve essere Item x Item
        if self.W_sparse.shape[0] != self.n_items:
             raise ValueError(f"Error: W_sparse size ({self.W_sparse.shape}) doesn't match n_items ({self.n_items}). "
                              "Check URM orientation.")

        if self.verbose:
            print(f"{self.RECOMMENDER_NAME}: Training complete. W_sparse shape: {self.W_sparse.shape}")

    def _compute_item_score(self, user_id_array, items_to_compute=None):
        user_profile_batch = self.URM_train[user_id_array]

        if items_to_compute is not None:
            scores = user_profile_batch.dot(self.W_sparse[:, items_to_compute]).toarray()
        else:
            scores = user_profile_batch.dot(self.W_sparse).toarray()

        return scores




TFIDF_recommender = ImplicitTFIDFRecommender(URM_fit_models, verbose=True)

TFIDF_recommender.fit(K=7, num_threads=0)

In [ ]:
#PureSVD
from sklearn.feature_extraction.text import TfidfTransformer
from Recommenders.MatrixFactorization.PureSVDRecommender import PureSVDRecommender
import numpy as np
import scipy.sparse as sps

def start_bm25_transformation(X, k1=1.2, b=0.75):
    """
    Applica la trasformazione Okapi BM25 a una matrice sparsa X (Utenti x Item).
    X: csr_matrix
    k1: controlla la saturazione della frequenza del termine (TF).
    b: controlla la normalizzazione della lunghezza del documento (profilo utente).
    """
    X = X.copy()

    N = X.shape[0]
    item_freqs = np.diff(X.tocsc().indptr)
    idf = np.log(N / (item_freqs + 1.0) + 1.0)

    doc_len = np.diff(X.indptr)
    avg_dl = doc_len.mean()

    per_doc_denom = k1 * (1 - b + b * (doc_len / avg_dl))

    per_doc_denom_expanded = np.repeat(per_doc_denom, np.diff(X.indptr))


    tf = X.data


    idf_expanded = idf[X.indices]

    new_data = idf_expanded * ((tf * (k1 + 1)) / (tf + per_doc_denom_expanded))

    X.data = new_data
    return X


k1 = 1.996904472023335
b = 0.12355408697503084
URM_fit_models_transformed = start_bm25_transformation(URM_fit_models, k1=k1, b=b)

PureSVD_recommender = PureSVDRecommender(URM_fit_models_transformed, verbose=True)

PureSVD_recommender.fit(
    num_factors=121,
    random_seed=1234
)

# Candidate Generators

In [ ]:
trained_models = {
    'slim': SLIM_EN_recommender,
    'slim_neg': SLIM_EN_neg_recommender,
    'rp3sharp': RP3BetaSharp_recommender,
    'rp3prec': RP3BetaHighPrec_recommender,
    'ials': iALS_recommender,
    'easer': EASE_R_recommender,
    'lightFM': lightFM_recommender,
    'multVAE': multVAE_recommender,
    'TFIDF': TFIDF_recommender,
    'PureSVD': PureSVD_recommender,
}


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

MODELS_TO_TEST = list(trained_models.keys())
MAX_CUTOFF = 250
TEST_USERS = np.unique(URM_test.indices)

print(f"Pre-calcolo raccomandazioni (Top {MAX_CUTOFF}) per {len(MODELS_TO_TEST)} modelli...")

rec_cache = {m: {} for m in MODELS_TO_TEST}

for name in MODELS_TO_TEST:
    model = trained_models[name]
    print(f" -> Caching {name}...")
    for user_id in tqdm(TEST_USERS, leave=False):
        try:
            recs = model.recommend(user_id, cutoff=MAX_CUTOFF)
            if isinstance(recs, tuple): recs = recs[0] # Gestione tuple
            rec_cache[name][user_id] = recs
        except:
            rec_cache[name][user_id] = []

print("Caching completato.")

In [ ]:
def evaluate_combination(selected_models_config, urm_ground_truth, test_users, cache):
    """
    Valuta una specifica combinazione di modelli e cutoff.
    config = [('rp3', 50), ('ials', 40), ...]
    """
    gt_csr = urm_ground_truth.tocsr()
    total_hits = 0
    total_candidates = 0
    total_relevant = 0

    for u in test_users:
        start = gt_csr.indptr[u]
        end = gt_csr.indptr[u+1]
        true_items = set(gt_csr.indices[start:end])

        if len(true_items) == 0: continue
        total_relevant += len(true_items)

        union_candidates = set()
        for name, cutoff in selected_models_config:
            items = cache[name][u][:cutoff]
            union_candidates.update(items)

        total_candidates += len(union_candidates)
        total_hits += len(union_candidates.intersection(true_items))

    recall = total_hits / total_relevant if total_relevant > 0 else 0
    avg_candidates = total_candidates / len(test_users)
    precision = total_hits / total_candidates if total_candidates > 0 else 0

    return recall, avg_candidates, precision


# Parametri
cutoffs_to_try = [20, 30, 40, 50, 60, 70, 80, 90, 100, 110, 120, 130, 140, 150] # Cutoff ragionevoli da testare

# Stato Iniziale
selected_sequence = []
current_best_recall = 0.0

history_data = []

print("\n--- INIZIO SELEZIONE GREEDY ---")

for round_idx in range(10):
    best_candidate_model = None
    best_candidate_cutoff = 0
    best_marginal_gain = -1
    best_new_metrics = (0,0,0) # recall, cands, prec

    print(f"\nROUND {round_idx + 1}: Cerco il miglior modello da aggiungere...")

    current_names = [x[0] for x in selected_sequence]
    available_models = [m for m in MODELS_TO_TEST if m not in current_names]

    if not available_models: break

    for model in available_models:
        for cutoff in cutoffs_to_try:
            temp_config = selected_sequence + [(model, cutoff)]

            recall, avg_cands, prec = evaluate_combination(temp_config, URM_test, TEST_USERS, rec_cache)

            gain = recall - current_best_recall

            if gain > best_marginal_gain:
                best_marginal_gain = gain
                best_candidate_model = model
                best_candidate_cutoff = cutoff
                best_new_metrics = (recall, avg_cands, prec)

    if best_marginal_gain < 0.001:
        print(f"STOP: Guadagno marginale troppo basso ({best_marginal_gain:.5f}).")
        break

    selected_sequence.append((best_candidate_model, best_candidate_cutoff))
    current_best_recall = best_new_metrics[0]

    print(f" >>> WINNER: {best_candidate_model} (Cutoff {best_candidate_cutoff})")
    print(f"     Gain: +{best_marginal_gain:.5f} | Tot Recall: {current_best_recall:.5f} | Avg Cands: {best_new_metrics[1]:.1f}")

    history_data.append({
        "Round": round_idx+1,
        "Added Model": best_candidate_model,
        "Cutoff": best_candidate_cutoff,
        "Cumulative Recall": current_best_recall,
        "Avg Candidates": best_new_metrics[1],
        "Precision": best_new_metrics[2]
    })

df_results = pd.DataFrame(history_data)
display(df_results)

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 6))

ax1.set_xlabel('Avg Candidates per User (Cost)')
ax1.set_ylabel('Theoretical Max Recall', color='blue')
ax1.plot(df_results['Avg Candidates'], df_results['Cumulative Recall'], 'b-o', linewidth=2, label='Recall')
ax1.tick_params(axis='y', labelcolor='blue')
ax1.grid(True)

for i, txt in enumerate(df_results['Added Model']):
    cutoff = df_results['Cutoff'].iloc[i]
    ax1.annotate(f"+ {txt} ({cutoff})",
                 (df_results['Avg Candidates'].iloc[i], df_results['Cumulative Recall'].iloc[i]),
                 xytext=(0, 10), textcoords='offset points', ha='center')

ax2 = ax1.twinx()
ax2.set_ylabel('Precision (Cleanliness)', color='green')
ax2.plot(df_results['Avg Candidates'], df_results['Precision'], 'g--s', alpha=0.6, label='Precision')
ax2.tick_params(axis='y', labelcolor='green')

plt.title("Candidate Generation Optimization Path")
plt.show()

# Cutoff 125

In [ ]:
def evaluate_combination(selected_models_config, urm_ground_truth, test_users, cache):
    """
    Valuta una specifica combinazione di modelli e cutoff.
    config = [('rp3', 50), ('ials', 40), ...]
    """
    gt_csr = urm_ground_truth.tocsr()
    total_hits = 0
    total_candidates = 0
    total_relevant = 0

    for u in test_users:
        start = gt_csr.indptr[u]
        end = gt_csr.indptr[u+1]
        true_items = set(gt_csr.indices[start:end])

        if len(true_items) == 0: continue
        total_relevant += len(true_items)

        union_candidates = set()
        for name, cutoff in selected_models_config:
            items = cache[name][u][:cutoff]
            union_candidates.update(items)

        total_candidates += len(union_candidates)
        total_hits += len(union_candidates.intersection(true_items))

    recall = total_hits / total_relevant if total_relevant > 0 else 0
    avg_candidates = total_candidates / len(test_users)
    precision = total_hits / total_candidates if total_candidates > 0 else 0

    return recall, avg_candidates, precision

# --- FASE 2: GREEDY OPTIMIZATION LOOP ---

cutoffs_to_try = [20, 30, 40, 50, 60, 70, 80, 90, 100, 110, 120, 125] # Cutoff ragionevoli da testare

# Stato Iniziale
selected_sequence = []
current_best_recall = 0.0

history_data = []

print("\n--- INIZIO SELEZIONE GREEDY ---")

for round_idx in range(10):
    best_candidate_model = None
    best_candidate_cutoff = 0
    best_marginal_gain = -1
    best_new_metrics = (0,0,0)

    print(f"\nROUND {round_idx + 1}: Cerco il miglior modello da aggiungere...")


    current_names = [x[0] for x in selected_sequence]
    available_models = [m for m in MODELS_TO_TEST if m not in current_names]

    if not available_models: break

    for model in available_models:
        for cutoff in cutoffs_to_try:
            temp_config = selected_sequence + [(model, cutoff)]

            recall, avg_cands, prec = evaluate_combination(temp_config, URM_test, TEST_USERS, rec_cache)

            gain = recall - current_best_recall


            if gain > best_marginal_gain:
                best_marginal_gain = gain
                best_candidate_model = model
                best_candidate_cutoff = cutoff
                best_new_metrics = (recall, avg_cands, prec)

    if best_marginal_gain < 0.001:
        print(f"STOP: Guadagno marginale troppo basso ({best_marginal_gain:.5f}).")
        break

    selected_sequence.append((best_candidate_model, best_candidate_cutoff))
    current_best_recall = best_new_metrics[0]

    print(f" >>> WINNER: {best_candidate_model} (Cutoff {best_candidate_cutoff})")
    print(f"     Gain: +{best_marginal_gain:.5f} | Tot Recall: {current_best_recall:.5f} | Avg Cands: {best_new_metrics[1]:.1f}")

    history_data.append({
        "Round": round_idx+1,
        "Added Model": best_candidate_model,
        "Cutoff": best_candidate_cutoff,
        "Cumulative Recall": current_best_recall,
        "Avg Candidates": best_new_metrics[1],
        "Precision": best_new_metrics[2]
    })

df_results = pd.DataFrame(history_data)
display(df_results)

# Cutoff Search

## 1. Baseline Analysis: Global Cutoff Evaluation
Before implementing personalized cutoffs, we establish a baseline by applying a uniform global cutoff ($K$) across all candidate generators.

By varying $K$ from 10 to 250, we track two conflicting metrics:
* **Max Recall Achieved (Recall Ceiling):** The upper bound of recall our XGBoost model can theoretically reach.
* **% Positives (Precision/Density):** The ratio of true positives to total candidates. A lower percentage means the XGBoost model will have to sift through more "noise", heavily impacting training time, memory usage, and class imbalance.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

cutoffs_to_benchmark = [20, 30, 40, 50, 65, 80, 100, 125]

sensitivity_results = []

print(f"--- INIZIO SENSITIVITY ANALYSIS SU {len(cutoffs_to_benchmark)} LIVELLI DI CUTOFF ---")

for global_cutoff in cutoffs_to_benchmark:
    print(f"\n>>> TEST CUTOFF GLOBALE: {global_cutoff}")

    selected_models = []
    current_recall = 0.0
    current_candidates = 0
    current_precision = 0.0

    while True:
        best_model = None
        best_gain = -1
        best_stats = (0,0,0)

        available = [m for m in MODELS_TO_TEST if m not in selected_models]

        if not available: break

        for model in available:
            temp_config = [(m, global_cutoff) for m in selected_models] + [(model, global_cutoff)]

            recall, avg_cands, prec = evaluate_combination(temp_config, URM_test, TEST_USERS, rec_cache)

            gain = recall - current_recall

            if gain > best_gain:
                best_gain = gain
                best_model = model
                best_stats = (recall, avg_cands, prec)

        if best_gain < 0.001:
            break

        selected_models.append(best_model)
        current_recall = best_stats[0]
        current_candidates = best_stats[1]
        current_precision = best_stats[2]


    print(f"   BEST SET ({len(selected_models)} models): {selected_models}")
    print(f"   Recall: {current_recall:.4f} | % Positives: {current_precision:.4%}")

    sensitivity_results.append({
        "Cutoff": global_cutoff,
        "Optimal Model Count": len(selected_models),
        "Selected Models": ", ".join(selected_models),
        "Max Recall": current_recall,
        "Avg Candidates": current_candidates,
        "% Positives": current_precision * 100, # In percentuale
        "F1 Score Proxy": (2 * current_recall * current_precision) / (current_recall + current_precision + 1e-9)
    })

df_sens = pd.DataFrame(sensitivity_results)

print("\n--- RISULTATI ANALISI SENSIBILITÀ ---")
display(df_sens.style.background_gradient(subset=['Max Recall', '% Positives', 'F1 Score Proxy'], cmap='viridis'))

fig, ax1 = plt.subplots(figsize=(12, 7))

ax1.set_xlabel('Global Cutoff (K)', fontsize=12)
ax1.set_ylabel('Max Recall Achieved', color='tab:blue', fontsize=12, fontweight='bold')
ax1.plot(df_sens['Cutoff'], df_sens['Max Recall'], color='tab:blue', marker='o', linewidth=3, label='Recall')
ax1.tick_params(axis='y', labelcolor='tab:blue')
ax1.grid(True, alpha=0.3)

ax2 = ax1.twinx()
ax2.set_ylabel('% Positives (Precision)', color='tab:red', fontsize=12, fontweight='bold')
ax2.plot(df_sens['Cutoff'], df_sens['% Positives'], color='tab:red', marker='s', linestyle='--', linewidth=3, label='% Positives')
ax2.tick_params(axis='y', labelcolor='tab:red')

plt.title("Candidate Generation: Impact of Cutoff on Quality vs Quantity", fontsize=14)

for i, txt in enumerate(df_sens['Optimal Model Count']):
    ax1.annotate(f"{txt} Models",
                 (df_sens['Cutoff'][i], df_sens['Max Recall'][i]),
                 xytext=(0, -15), textcoords='offset points', ha='center', fontsize=9, color='black')

plt.tight_layout()
plt.show()

# Complete Analysis

## 2. Advanced Retrieval: Marginal Precision Thresholding
To optimize the candidate pool, we move away from global cutoffs. Here, we define the functions to compute the **Marginal Precision** of each recommender.

* Iterate through ranks $1$ to $N$.
* Calculate the precision specifically at rank $k$.
* Stop extracting candidates from a model when adding the next item brings less value than our predefined noise tolerance (Target Precision).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

cutoffs_to_benchmark = [20, 30, 40, 50, 65, 80, 100, 125, 150, 175, 200, 225, 250]
sensitivity_results = []

print(f"--- INIZIO SENSITIVITY ANALYSIS DETTAGLIATA SU {len(cutoffs_to_benchmark)} LIVELLI ---")

for global_cutoff in cutoffs_to_benchmark:
    print(f"\n{'='*60}")
    print(f" >>> START TRIAL: GLOBAL CUTOFF = {global_cutoff}")
    print(f"{'='*60}")

    selected_models = []
    current_recall = 0.0
    current_candidates = 0
    current_precision = 0.0

    round_num = 1

    # Greedy Loop
    while True:
        best_model = None
        best_gain = -1
        best_stats = (0,0,0) # recall, cands, prec

        available = [m for m in MODELS_TO_TEST if m not in selected_models]

        if not available:
            print("   (Tutti i modelli selezionati)")
            break

        for model in available:
            temp_config = [(m, global_cutoff) for m in selected_models] + [(model, global_cutoff)]

            recall, avg_cands, prec = evaluate_combination(temp_config, URM_test, TEST_USERS, rec_cache)

            gain = recall - current_recall

            if gain > best_gain:
                best_gain = gain
                best_model = model
                best_stats = (recall, avg_cands, prec)

        if best_gain < 0.0005:
            print(f"   [STOP] Guadagno troppo basso (+{best_gain:.5f}). Mi fermo qui.")
            break

        selected_models.append(best_model)
        current_recall = best_stats[0]
        current_candidates = best_stats[1]
        current_precision = best_stats[2]

        print(f"   Round {round_num}: + {best_model:<15}")
        print(f"       Recall: {current_recall:.5f} (Gain: +{best_gain:.5f})")
        print(f"       Avg Cands: {current_candidates:.1f} | % Positives: {current_precision:.4%}")
        print(f"       ------------------------------------------------")

        round_num += 1

    print(f"   >>> TRIAL COMPLETO. Best Set: {selected_models}")

    sensitivity_results.append({
        "Cutoff": global_cutoff,
        "Optimal Model Count": len(selected_models),
        "Selected Models": ", ".join(selected_models),
        "Max Recall": current_recall,
        "Avg Candidates": current_candidates,
        "% Positives": current_precision * 100,
        "F1 Score Proxy": (2 * current_recall * current_precision) / (current_recall + current_precision + 1e-9)
    })

df_sens = pd.DataFrame(sensitivity_results)

print("\n\n")
print(f"{'#'*80}")
print(f" RIASSUNTO FINALE SENSITIVITY ANALYSIS")
print(f"{'#'*80}")
display(df_sens.style.background_gradient(subset=['Max Recall', '% Positives', 'F1 Score Proxy'], cmap='viridis'))

fig, ax1 = plt.subplots(figsize=(12, 7))

ax1.set_xlabel('Global Cutoff (K)', fontsize=12)
ax1.set_ylabel('Max Recall Achieved', color='tab:blue', fontsize=12, fontweight='bold')
ax1.plot(df_sens['Cutoff'], df_sens['Max Recall'], color='tab:blue', marker='o', linewidth=3, label='Recall')
ax1.tick_params(axis='y', labelcolor='tab:blue')
ax1.grid(True, alpha=0.3)

ax2 = ax1.twinx()
ax2.set_ylabel('% Positives (Precision)', color='tab:red', fontsize=12, fontweight='bold')
ax2.plot(df_sens['Cutoff'], df_sens['% Positives'], color='tab:red', marker='s', linestyle='--', linewidth=3, label='% Positives')
ax2.tick_params(axis='y', labelcolor='tab:red')

plt.title("Candidate Generation Strategy: Impact of Cutoff", fontsize=14)

for i, txt in enumerate(df_sens['Optimal Model Count']):
    ax1.annotate(f"{txt} Models",
                 (df_sens['Cutoff'][i], df_sens['Max Recall'][i]),
                 xytext=(0, -15), textcoords='offset points', ha='center', fontsize=9, color='black')

plt.tight_layout()
plt.show()

# Overlap e Group

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

def analyze_overlap_and_groups(models_dict, URM_train, URM_test, cutoffs_list):
    """
    Analisi completa:
    1. Sovrapposizione candidati (Jaccard Similarity)
    2. Performance per gruppo utente (Cold vs Warm)
    """

    profile_lengths = np.ediff1d(URM_train.indptr)

    sorted_users = np.argsort(profile_lengths)
    n_users = len(sorted_users)
    group_size = int(n_users / 5)

    user_groups = {}
    group_labels = []

    for i in range(5):
        start = i * group_size
        end = (i + 1) * group_size if i < 4 else n_users
        u_group = sorted_users[start:end]

        avg_len = np.mean(profile_lengths[u_group])
        label = f"Grp {i+1} (Avg: {avg_len:.0f})"
        user_groups[label] = u_group
        group_labels.append(label)

    test_users_sample = np.unique(URM_test.indices)
    if len(test_users_sample) > 2000:
        test_users_sample = np.random.choice(test_users_sample, 2000, replace=False)

    model_names = list(models_dict.keys())

    for k in cutoffs_list:
        print(f"\n{'='*60}")
        print(f" ANALISI PER CUTOFF K = {k}")
        print(f"{'='*60}")

        # OVERLAP ANALYSIS (Jaccard Heatmap)
        print("Calcolo Overlap (Jaccard)...")
        jaccard_matrix = np.zeros((len(model_names), len(model_names)))

        recs_sample = {m: [] for m in model_names}
        for name in model_names:
            model = models_dict[name]
            for u in test_users_sample:
                try:
                    r = model.recommend(u, cutoff=k)
                    if isinstance(r, tuple): r = r[0]
                    recs_sample[name].append(set(r))
                except:
                    recs_sample[name].append(set())

        for i, m1 in enumerate(model_names):
            for j, m2 in enumerate(model_names):
                if i > j: continue # Simmetrica
                if i == j:
                    jaccard_matrix[i, j] = 1.0
                    continue

                jaccard_sum = 0
                count = 0
                for idx_u in range(len(test_users_sample)):
                    set1 = recs_sample[m1][idx_u]
                    set2 = recs_sample[m2][idx_u]

                    if len(set1) == 0 and len(set2) == 0: continue

                    intersection = len(set1.intersection(set2))
                    union = len(set1.union(set2))

                    if union > 0:
                        jaccard_sum += intersection / union
                        count += 1

                score = jaccard_sum / count if count > 0 else 0
                jaccard_matrix[i, j] = score
                jaccard_matrix[j, i] = score

        plt.figure(figsize=(10, 8))
        sns.heatmap(jaccard_matrix, annot=True, fmt=".2f",
                    xticklabels=model_names, yticklabels=model_names,
                    cmap="YlGnBu", vmin=0, vmax=1)
        plt.title(f"Model Overlap (Jaccard Similarity) @ K={k}")
        plt.show()


        print("Calcolo Performance per Gruppo Utenti...")

        group_results = {m: [] for m in model_names}

        for label in group_labels:
            users_in_group = user_groups[label]
            valid_users = np.intersect1d(users_in_group, np.unique(URM_test.indices))

            if len(valid_users) == 0:
                for m in model_names: group_results[m].append(0)
                continue


            gt_csr = URM_test.tocsr()

            for name in model_names:
                model = models_dict[name]
                hits = 0
                total_relevant = 0

                eval_users = valid_users if len(valid_users) < 1000 else np.random.choice(valid_users, 1000, replace=False)

                for u in eval_users:
                    start = gt_csr.indptr[u]
                    end = gt_csr.indptr[u+1]
                    true_items = gt_csr.indices[start:end]

                    if len(true_items) == 0: continue
                    total_relevant += len(true_items)

                    try:
                        rec_list = model.recommend(u, cutoff=k)
                        if isinstance(rec_list, tuple): rec_list = rec_list[0]

                        hits += np.isin(rec_list, true_items).sum()
                    except: pass

                recall = hits / total_relevant if total_relevant > 0 else 0
                group_results[name].append(recall)

        plt.figure(figsize=(12, 6))
        markers = ['o', 's', '^', 'D', 'v', '<', '>', 'p', '*', 'h']

        for i, name in enumerate(model_names):
            plt.plot(group_labels, group_results[name],
                     marker=markers[i % len(markers)], linewidth=2, label=name)

        plt.title(f"Recall@{k} by User Profile Length (Cold to Warm)")
        plt.xlabel("User Group (Avg Interactions)")
        plt.ylabel(f"Recall@{k}")
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.show()

# --- ESECUZIONE ---
requested_cutoffs = [75, 125, 150, 200, 250]

analyze_overlap_and_groups(trained_models, URM_fit_models, URM_test, requested_cutoffs)

## 3. Visualizing Precision Decay Across Models
The following plots illustrate the core of our dynamic cutoff strategy.

* The **Blue Line** represents the marginal precision decay as the rank increases.
* The **Red Dashed Line** represents our target precision threshold (e.g., 4.0%).
* The **Green Vertical Line** indicates the optimal cutoff point for the specific model.

Notice how models like **SLIM ElasticNet** maintain high precision much deeper into the ranking compared to **LightFM**, justifying a significantly larger candidate allocation for SLIM.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

def analyze_marginal_precision(recommender_object, URM_test, max_cutoff=200, bin_size=10, min_precision_threshold=0.01, make_plot=True):
    """
    Calcola dove la precisione marginale del modello crolla sotto la soglia.
    """
    n_users = URM_test.shape[0]

    try:
        recs = recommender_object.recommend(np.arange(n_users), cutoff=max_cutoff)
    except:
        recs = []
        for start in range(0, n_users, 2000):
            end = min(start + 2000, n_users)
            recs.extend(recommender_object.recommend(np.arange(start, end), cutoff=max_cutoff))

    recs = np.array(recs)

    hits_matrix = np.zeros(recs.shape, dtype=int)
    URM_test_csr = URM_test.tocsr()

    for u in range(n_users):
        start = URM_test_csr.indptr[u]
        end = URM_test_csr.indptr[u+1]
        if end > start:
            true_items = URM_test_csr.indices[start:end]
            hits_matrix[u, :] = np.isin(recs[u, :], true_items)

    marginal_precisions = []
    cutoffs = []
    optimal_cutoff = max_cutoff
    found_drop = False

    for k in range(0, max_cutoff, bin_size):
        start_pos = k
        end_pos = min(k + bin_size, max_cutoff)

        hits_in_bin = hits_matrix[:, start_pos:end_pos].sum()
        slots_in_bin = n_users * (end_pos - start_pos)

        prec = hits_in_bin / slots_in_bin if slots_in_bin > 0 else 0
        marginal_precisions.append(prec)
        cutoffs.append(end_pos)

        if not found_drop and prec < min_precision_threshold:
            optimal_cutoff = start_pos
            found_drop = True

    if make_plot:
        plt.figure(figsize=(8, 4))
        plt.plot(cutoffs, marginal_precisions, marker='o', label='Marginal Precision')
        plt.axhline(y=min_precision_threshold, color='r', linestyle='--', label=f'Threshold ({min_precision_threshold:.1%})')
        plt.axvline(x=optimal_cutoff, color='g', linestyle='--', label=f'Opt Cutoff: {optimal_cutoff}')
        plt.title(f'{recommender_object.RECOMMENDER_NAME} (Opt: {optimal_cutoff})')
        plt.xlabel('Rank')
        plt.ylabel('Precision')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.show()

    return optimal_cutoff, marginal_precisions[0]


ANALYSIS_MAX_CUTOFF = 250
ANALYSIS_BIN_SIZE = 10
THRESHOLD = 0.04

final_suggestions = []

print(f"---ANALISI AUTOMATICA CUTOFF (Soglia: {THRESHOLD:.1%}) ---\n")

for name, model in trained_models.items():
    print(f">>> Analisi Modello: {name} ...")

    opt_k, start_prec = analyze_marginal_precision(
        model,
        URM_test,
        max_cutoff=ANALYSIS_MAX_CUTOFF,
        bin_size=ANALYSIS_BIN_SIZE,
        min_precision_threshold=THRESHOLD,
        make_plot=True
    )

    final_suggestions.append({
        "Model": name,
        "Optimal Cutoff": opt_k,
        "Initial Precision": start_prec,
        "Status": "Good" if opt_k > 20 else "Weak"
    })


df_suggestions = pd.DataFrame(final_suggestions).sort_values("Optimal Cutoff", ascending=False)

print("\n\n--- TABELLA SUGGERIMENTI CUTOFF ---")
def highlight_cutoff(val):
    color = 'green' if val >= 50 else 'orange' if val >= 30 else 'red'
    return f'color: {color}; font-weight: bold'

display(df_suggestions.style.applymap(highlight_cutoff, subset=['Optimal Cutoff']))

print("\n>>> DIZIONARIO SUGGERITO PER GENERAZIONE CANDIDATI:")
print("generators_dict = {")
for _, row in df_suggestions.iterrows():
    if row['Optimal Cutoff'] > 0:
        print(f"    '{row['Model']}': ({row['Model']}, {row['Optimal Cutoff']}),")
print("}")

## 4. Conclusion: Selecting the Optimal Retrieval Configuration

By analyzing the tradeoff curve across different precision thresholds (from 1% to 10%), we can identify the "sweet spot" for our XGBoost reranker.

**Final Architectural Decision:**
Based on the memory limits and the computational budget for the competition, we selected a **target marginal precision of ~2.0% - 3.0%**.

This specific threshold provides the best of both worlds:
1. It achieves a good **Recall Ceiling** (> 0.70).
2. It limits the average candidate set size per user to a highly manageable number, preventing memory overflow during the XGBoost DMatrix construction and allowing for ultra-fast Numba feature engineering in the next stage.